# PromptPiper — Results (logged runs)

Figures, tables and paper numbers for runs produced by the compressor with complete logging
(result JSON + `{run}_log/`). All computations are in `prompt_piper.analysis` (unit-tested);
this notebook only plots and prints.

Differences from `visualizations.ipynb` (which reproduces the submitted paper):
* the **compromise point is selected on the search split** (`SELECTION = "search"`); set `"test"` to reproduce the old selection;
* **NSGA-II, Random Search and MAB are each represented by their own compromise point** (same rule), not RS/MAB front means;
* exact hypervolume (pymoo), no silent fallback from test to search values;
* per-sample data, cost and convergence come from the logs of the same run (no separate `detailed_eval` pass).

Configure with environment variables: `PP_RESULTS_DIR` (default `../results/icse_26_final`),
`PP_FIG_DIR` (default `../paper/prompt_piper_paper/figures/final`), `PP_SELECTION`.

In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import binomtest, wilcoxon

from prompt_piper.analysis import (
    METHOD_LABELS, compare_methods, convergence, cost, standalone_cost, front_hypervolume, load_runs,
    load_transfers, method_points, operator_frequency, per_sample, select_compromise,
)
from prompt_piper.tools.validate_run import validate_run

warnings.filterwarnings("ignore")
RESULTS_DIR = Path(os.environ.get("PP_RESULTS_DIR", "../results/icse_26_final"))
FIG_DIR = Path(os.environ.get("PP_FIG_DIR", "../paper/prompt_piper_paper/figures/final"))
SELECTION = os.environ.get("PP_SELECTION", "search")
FIG_DIR.mkdir(parents=True, exist_ok=True)
COL1, COL2, ROW_H = 6.3, 13, 2.5

runs = load_runs(RESULTS_DIR)
MODELS = sorted({r.model for r in runs})
TASKS = sorted({r.name for r in runs})
print(f"{len(runs)} runs | models: {MODELS} | tasks: {TASKS} | selection on: {SELECTION}")

## Completeness check
Every run must pass log validation (no missing data points) before its results are used.

In [ ]:
checks = pd.DataFrame([
    {"model": r.model, "task": r.name, **{k: v for k, v in validate_run(r.log_dir).items() if k in ("ok", "n_errors")}}
    for r in runs
])
display(checks)
assert checks["ok"].all(), "Some runs have incomplete logs — see validate_run output."


---
## RQ1 — Trade-off between compression and performance

In [ ]:
STYLE = {
    "no_compression": ("*", "black"), "all_transformations": ("X", "tab:red"),
    "random_search": ("s", "tab:orange"), "multi_armed_bandit": ("D", "tab:purple"),
    "summarization": ("P", "tab:brown"), "truncated_prompt": ("v", "tab:pink"),
    "random_deletion": ("^", "tab:olive"), "llmlingua2": ("<", "tab:cyan"),
}
fig, axes = plt.subplots(len(MODELS), len(TASKS), figsize=(COL2, ROW_H * len(MODELS)),
                         squeeze=False, sharey=True)
for r in runs:
    ax = axes[MODELS.index(r.model)][TASKS.index(r.name)]
    for split, style, color in (("search", "o--", "lightsteelblue"), ("test", "o-", "navy")):
        pts = sorted((s[f"{split}_avg_tokens"], s[f"{split}_avg_performance"]) for s in r.solutions(r.algorithm))
        ax.plot(*zip(*pts), style, color=color, ms=3, label=f"NSGA-II ({split})")
    i, sol = select_compromise(r, r.algorithm, SELECTION)
    ax.scatter(sol["test_avg_tokens"], sol["test_avg_performance"], s=120, facecolors="none",
               edgecolors="navy", linewidths=2, label="NSGA-II compromise", zorder=6)
    for b in r.data["baselines"]:
        marker, color = STYLE.get(b["name"], ("o", "gray"))
        if b["name"] in ("random_search", "multi_armed_bandit"):
            _, bs = select_compromise(r, b["name"], SELECTION)
            ax.scatter(bs["test_avg_tokens"], bs["test_avg_performance"], marker=marker, color=color,
                       s=60, label=METHOD_LABELS[b["name"]], zorder=5)
        else:
            ax.scatter(b["test_avg_tokens"], b["test_avg_performance"], marker=marker, color=color,
                       s=60, label=METHOD_LABELS.get(b["name"], b["name"]), zorder=5)
    ax.set_title(f"{r.model}\n{r.name}", fontsize=8)
    ax.set_ylim(-0.05, 1.05)
handles, labels = axes[0][0].get_legend_handles_labels()
fig.supxlabel("Avg tokens (test)", y=0.09); fig.supylabel("Avg performance (test)")
fig.legend(handles, labels, loc="lower center", ncol=6, fontsize=8)
fig.tight_layout(rect=(0, 0.1, 1, 1))
fig.savefig(FIG_DIR / "rq1_pareto_grid.pdf", bbox_inches="tight")

### RQ1 — per-run tests: compromise vs uncompressed prompt (paired over test samples)

In [ ]:
rows = []
for r in runs:
    i, _ = select_compromise(r, r.algorithm, SELECTION)
    a = per_sample(r, r.algorithm, i)
    b = per_sample(r, "no_compression")
    m = a.merge(b, on="sample_id", suffixes=("_c", "_nc"))
    binary = set(np.unique(m[["score_c", "score_nc"]].to_numpy())) <= {0.0, 1.0}
    if binary:
        n01 = int(((m.score_c == 1) & (m.score_nc == 0)).sum()); n10 = int(((m.score_c == 0) & (m.score_nc == 1)).sum())
        p_perf = binomtest(min(n01, n10), n01 + n10).pvalue if n01 + n10 else 1.0
    else:
        p_perf = wilcoxon(m.score_c, m.score_nc).pvalue if (m.score_c != m.score_nc).any() else 1.0
    rows.append({
        "model": r.model, "task": r.name, "n": len(m),
        "perf_compromise": m.score_c.mean(), "perf_uncompressed": m.score_nc.mean(),
        "delta_perf": m.score_c.mean() - m.score_nc.mean(),
        "test": "exact McNemar" if binary else "Wilcoxon", "p_perf": p_perf,
        "tokens_compromise": m.tokens_c.mean(), "tokens_uncompressed": m.tokens_nc.mean(),
        "token_reduction": 1 - m.tokens_c.mean() / m.tokens_nc.mean(),
        "p_tokens": wilcoxon(m.tokens_c, m.tokens_nc).pvalue if (m.tokens_c != m.tokens_nc).any() else 1.0,
    })
df_rq1 = pd.DataFrame(rows)
display(df_rq1.round(3))
df_rq1.to_csv(FIG_DIR / "rq1_statistical_tests.csv", index=False)

---
## RQ2 — Comparison with baselines (points)

In [ ]:
points = method_points(runs, selection=SELECTION)
assert not points["legacy_front_mean"].any(), "Legacy results: RS/MAB solutions not saved."
summary = (points.groupby("method")[["test_performance", "test_tokens", "compression", "utility"]]
           .mean().rename(index=METHOD_LABELS).sort_values("test_performance", ascending=False))
display(summary.round(3))
omni, post = [], []
for metric, hib in (("test_performance", True), ("compression", True), ("utility", True)):
    o, p = compare_methods(points, metric, higher_is_better=hib)
    omni.append(o); post.append(p)
df_omni, df_post = pd.DataFrame(omni), pd.concat(post, ignore_index=True)
display(df_omni.round(4)); display(df_post.round(4))
df_omni.to_csv(FIG_DIR / "rq2_friedman_tests.csv", index=False)
df_post.to_csv(FIG_DIR / "rq2_holm_wilcoxon_posthoc.csv", index=False)

In [ ]:
# Per-sample distributions of every method's point, pooled across models, per task.
per = []
for r in runs:
    for _, row in points[(points.model == r.model) & (points.task == r.name)].iterrows():
        d = per_sample(r, row.method, int(row.solution_index))
        d["task"], d["method"] = r.name, row.label
        per.append(d)
per = pd.concat(per, ignore_index=True)
order = list(summary.index)
fig, axes = plt.subplots(len(TASKS), 2, figsize=(COL1 * 1.5, ROW_H * len(TASKS)), squeeze=False)
for ti, task in enumerate(TASKS):
    sub = per[per.task == task]
    means = [sub[sub.method == m].score.mean() for m in order]
    ci = [1.96 * sub[sub.method == m].score.std(ddof=1) / np.sqrt(max((sub.method == m).sum(), 1)) for m in order]
    axes[ti][0].bar(range(len(order)), means, yerr=ci, capsize=3, color="steelblue")
    axes[ti][1].violinplot([sub[sub.method == m].tokens for m in order], showmedians=True)
    axes[ti][0].set_ylabel(task, fontsize=8)
for ax in axes[-1]:
    ax.set_xticks(range(len(order)) if ax is axes[-1][0] else range(1, len(order) + 1))
    ax.set_xticklabels(order, rotation=60, ha="right", fontsize=7)
axes[0][0].set_title("Performance (95% CI)"); axes[0][1].set_title("Tokens")
fig.tight_layout()
fig.savefig(FIG_DIR / "rq2_distributions.pdf", bbox_inches="tight")

---
## RQ3 — Cost and convergence

In [ ]:
df_cost = pd.DataFrame([cost(r) for r in runs])
nsga = points[points.method == "nsga2"].set_index(["model", "task"])
nc = points[points.method == "no_compression"].set_index(["model", "task"])
df_cost = df_cost.set_index(["model", "task"])
df_cost["saved_tokens_per_query"] = nc.test_tokens - nsga.test_tokens
df_cost["break_even_queries"] = df_cost.search_input_tokens / df_cost.saved_tokens_per_query
display(df_cost.round(1))
print("Break-even (input tokens) median:", df_cost.break_even_queries.median())
df_cost.to_csv(FIG_DIR / "rq3_cost.csv")
time_cols = [c for c in df_cost.columns if c.startswith("seconds_")]
ax = (df_cost[time_cols] / 3600).plot.bar(stacked=True, figsize=(COL2, 3), legend=False)
ax.set_ylabel("hours"); ax.legend(fontsize=6, ncol=4)
plt.tight_layout(); plt.savefig(FIG_DIR / "rq3_time_by_phase.pdf", bbox_inches="tight")

# Calls each search method needs on its own: methods share one evaluation cache per run,
# so logged calls and wall-clock times understate the cost of methods that run later.
df_standalone = pd.concat([standalone_cost(r) for r in runs], ignore_index=True)
display(df_standalone.groupby("method")[["candidates", "distinct_prompt_sets",
                                         "standalone_llm_calls", "llm_calls_made_in_run"]].mean().round(1))
df_standalone.to_csv(FIG_DIR / "rq3_standalone_cost.csv", index=False)

In [ ]:
conv = []
for r in runs:
    c = convergence(r); c["model"], c["task"] = r.model, r.name
    conv.append(c)
conv = pd.concat(conv, ignore_index=True)
fig, axes = plt.subplots(1, len(TASKS), figsize=(COL2, 2.8), squeeze=False, sharey=True)
for ti, task in enumerate(TASKS):
    ax = axes[0][ti]
    for method, g in conv[conv.task == task].groupby("method"):
        curve = g.groupby("n_visited").hypervolume.agg(["mean", "std"]).reset_index()
        ax.plot(curve.n_visited, curve["mean"], label=METHOD_LABELS.get(method, method))
        ax.fill_between(curve.n_visited, curve["mean"] - curve["std"].fillna(0),
                        curve["mean"] + curve["std"].fillna(0), alpha=0.2)
    ax.set_title(task, fontsize=8); ax.set_xlabel("evaluated strategies")
axes[0][0].set_ylabel("search-set hypervolume"); axes[0][0].legend(fontsize=7)
fig.tight_layout(); fig.savefig(FIG_DIR / "rq3_convergence.pdf", bbox_inches="tight")

final = conv.groupby(["model", "task", "method"]).hypervolume.transform("max")
conv["reached_95"] = conv.hypervolume >= 0.95 * final
first95 = (conv[conv.reached_95].groupby(["model", "task", "method"]).step.min().rename("step_95pct_hv"))
display(first95.unstack("method"))
first95.to_csv(FIG_DIR / "rq3_convergence_descriptors.csv")

hv = pd.DataFrame([{"model": r.model, "task": r.name, "method": m, "test_hv": front_hypervolume(r, m)}
                   for r in runs for m in ("nsga2", "random_search", "multi_armed_bandit") if r.solutions(m)])
display(hv.pivot_table(index=["model", "task"], columns="method", values="test_hv").round(3))

---
## RQ4 — Operator patterns and transferability

In [ ]:
freq = operator_frequency(runs)
fig, axes = plt.subplots(1, 3, figsize=(COL2, 6), sharey=True)
for ax, section in zip(axes, ("instruction", "shots", "data")):
    m = freq[freq.section == section].pivot_table(index="operator", columns="task", values="frequency", aggfunc="mean").fillna(0)
    im = ax.imshow(m.to_numpy(), aspect="auto", vmin=0, vmax=1, cmap="Blues")
    ax.set_title(section); ax.set_xticks(range(m.shape[1])); ax.set_xticklabels(m.columns, rotation=60, ha="right", fontsize=7)
    ax.set_yticks(range(m.shape[0])); ax.set_yticklabels(m.index, fontsize=7)
fig.colorbar(im, ax=axes, shrink=0.6, label="share of Pareto solutions")
fig.savefig(FIG_DIR / "rq4_2_operators.pdf", bbox_inches="tight")
freq.to_csv(FIG_DIR / "rq4_operator_frequency.csv", index=False)

In [ ]:
transfers = load_transfers(RESULTS_DIR)
if transfers.empty:
    print("No transfer results.")
else:
    native = pd.DataFrame([{"model": r.model, "target": r.name,
                            "native_tokens": np.mean([s["test_avg_tokens"] for s in r.solutions(r.algorithm)]),
                            "native_performance": np.mean([s["test_avg_performance"] for s in r.solutions(r.algorithm)])}
                           for r in runs])
    t = (transfers.groupby(["model", "source", "target"])[["test_tokens", "test_performance"]].mean()
         .reset_index().merge(native, on=["model", "target"]))
    display(t.round(3)); t.to_csv(FIG_DIR / "rq4_transfer.csv", index=False)

---
## Numbers for the paper

In [ ]:
s = summary
print(f"Selection on: {SELECTION}")
print(f"NSGA-II compromise: perf {s.loc['NSGA-II','test_performance']:.3f} at {s.loc['NSGA-II','test_tokens']:.0f} tokens "
      f"vs no compression {s.loc['No compression','test_performance']:.3f} at {s.loc['No compression','test_tokens']:.0f} "
      f"({1 - s.loc['NSGA-II','test_tokens'] / s.loc['No compression','test_tokens']:.1%} fewer tokens)")
print("Per-task NSGA-II vs no compression (mean over models):")
display(points[points.method.isin(["nsga2", "no_compression"])]
        .pivot_table(index="task", columns="method", values=["test_performance", "test_tokens"]).round(3))